In [1]:
import random
import string
import pandas as pd
import numpy as np
from datasets import load_dataset
import json
from huggingface_hub import login
import os
import matplotlib.pyplot as plt
from PIL import Image
import math
import stanza

from vqa_evaluation_prompts import *
from vqa_evaluator import VQAEvaluator
vqa_evaluator = VQAEvaluator()

# GPT Zero shot
PREDICTION_FOLDER_PATH = "results/gpt"
LABEL_FOLDER_PATH = "../../Annotations"
IMAGE_FOLDER = "../../10k_images"
print(PREDICTION_FOLDER_PATH)
print(LABEL_FOLDER_PATH)
print(IMAGE_FOLDER)

label_file = os.path.join(LABEL_FOLDER_PATH, "random1_vqa_references.json")
prediction_file = os.path.join(PREDICTION_FOLDER_PATH, "gpt_random1_vqa_seed1.json")

/home/xuezheng/anaconda3/envs/evaluations/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


results/gpt
../../Annotations
../../10k_images


# GPT4 zero shot

## Multi-label classification

In [2]:
y_pred, y_true, _ = vqa_evaluator.multilabel_classification_score(prediction_file, label_file)

0000001
{'0': 'No violations'}
0000002
{'1': {'reason': 'There is a person without a hard hat and the clothes do not cover shoulders and legs completely, and no high-visibility retroreflective vests are visible.', 'bounding_box': [0.25, 0.5, 0.35, 0.6]}, '4': {'reason': 'There are multiple workers in close proximity to the operating excavator, potentially in the blind spot of the operator.', 'bounding_box': [0.5, 0.6, 0.8, 0.9]}}
0000005
{'1': {'reason': 'The person on foot at the construction site is not wearing a hard hat, and the clothes do not cover the shoulders and legs completely. Shoes are not visible in the image.', 'bounding_box': [0.15, 0.55, 0.25, 0.95]}, '4': {'reason': 'The worker is walking in the potential blind spot of the excavator operator and within the operation radius of the excavator bucket.', 'bounding_box': [0.15, 0.55, 0.25, 0.95]}}
0000007
{'1': {'reason': "One worker is not wearing a hard hat, and another's clothes do not cover shoulders.", 'bounding_box': [

## Explanation

In [4]:
correctly_predicted = vqa_evaluator.correctly_predicted_images(prediction_file, label_file, isGeneratedFile=True)
print(correctly_predicted)
print(len(correctly_predicted['rule1']))
print(len(correctly_predicted['rule2']))
print(len(correctly_predicted['rule3']))
print(len(correctly_predicted['rule4']))

0000001
{'0': 'No violations'}
0000002
{'1': {'reason': 'There is a person without a hard hat and the clothes do not cover shoulders and legs completely, and no high-visibility retroreflective vests are visible.', 'bounding_box': [0.25, 0.5, 0.35, 0.6]}, '4': {'reason': 'There are multiple workers in close proximity to the operating excavator, potentially in the blind spot of the operator.', 'bounding_box': [0.5, 0.6, 0.8, 0.9]}}
0000005
{'1': {'reason': 'The person on foot at the construction site is not wearing a hard hat, and the clothes do not cover the shoulders and legs completely. Shoes are not visible in the image.', 'bounding_box': [0.15, 0.55, 0.25, 0.95]}, '4': {'reason': 'The worker is walking in the potential blind spot of the excavator operator and within the operation radius of the excavator bucket.', 'bounding_box': [0.15, 0.55, 0.25, 0.95]}}
0000007
{'1': {'reason': "One worker is not wearing a hard hat, and another's clothes do not cover shoulders.", 'bounding_box': [

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, Conversation, BitsAndBytesConfig, set_seed
from evaluate import load
import torch
import json
import os
import ast
import random
from vqa_evaluation_prompts import *

set_seed(20)
quantization_config = BitsAndBytesConfig(load_in_8bit=True)
model = AutoModelForCausalLM.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct", quantization_config=quantization_config, device_map="auto", num_beams = 5)
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct", quantization_config=quantization_config, device_map="auto")

/home/xuezheng/anaconda3/envs/llama3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.32it/s]
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [3]:
prompt_dict = {
    "system_prompt": [SYSTEM_PROMPT_RULE_1, SYSTEM_PROMPT_RULE_2, SYSTEM_PROMPT_RULE_3, SYSTEM_PROMPT_RULE_4],
    "user_prompt": [FEW_SHOT_PROMPT, USER_PROMPT_FINAL],
    "example_prompt": [[EXAMPLE_EVAL_RULE1_PROMPT_0000545, EXAMPLE_EVAL_RULE1_PROMPT_0000007, EXAMPLE_EVAL_RULE1_PROMPT_0000019], 
                       [EXAMPLE_EVAL_RULE2_PROMPT_0000925, EXAMPLE_EVAL_RULE2_PROMPT_0003632, EXAMPLE_EVAL_RULE2_PROMPT_0004235], 
                       [EXAMPLE_EVAL_RULE3_PROMPT_0001597, EXAMPLE_EVAL_RULE3_PROMPT_0000007, EXAMPLE_EVAL_RULE3_PROMPT_0000117], 
                       [EXAMPLE_EVAL_RULE4_PROMPT_0001512, EXAMPLE_EVAL_RULE4_PROMPT_0004725, EXAMPLE_EVAL_RULE4_PROMPT_0002093]]
}
folder_to_evaluate = "GPT_correctly_predicted"
all_files = os.listdir(folder_to_evaluate)
all_files = sorted(all_files)
prediction_files = [os.path.join(folder_to_evaluate, file) for file in all_files if file.endswith('.json')]

final_mark_dict = {}
i = 0
for file in prediction_files:
    print(file)
    mark_dict = {}
    
    with open(file, 'r') as infile:
        correctly_predicted = json.load(infile)
    id_list = sorted(list(correctly_predicted.keys()))

    for instance in id_list:
        print(instance)

        chatbot = pipeline(task="conversational", model=model, tokenizer=tokenizer)

        conversation = Conversation([{"role": "system", "content": prompt_dict['system_prompt'][i]}])
        conversation = chatbot(conversation)
        conversation.add_message({"role": "user", "content": prompt_dict['user_prompt'][0]})
        conversation = chatbot(conversation)

        conversation.add_message({"role": "user", "content": prompt_dict['example_prompt'][i][0]})
        conversation = chatbot(conversation)
        conversation.add_message({"role": "user", "content": prompt_dict['example_prompt'][i][1]})
        conversation = chatbot(conversation)
        conversation.add_message({"role": "user", "content": prompt_dict['example_prompt'][i][2]})
        conversation = chatbot(conversation)

        conversation.add_message({"role": "user", "content": f"Please evaluate the following reasoning:\n\n Candidate reasoning: {correctly_predicted[instance]['candidate']}\n\n Reference reasoning{correctly_predicted[instance]['reference']}"})
        conversation = chatbot(conversation)
        reply = conversation.messages[-1]["content"]
        print(correctly_predicted[instance]['candidate'])
        print(correctly_predicted[instance]['reference'])

        conversation.add_message({"role": "user", "content": prompt_dict['user_prompt'][1]})
        conversation = chatbot(conversation)
        llama_eval = conversation.messages[-1]["content"]
        llama_eval_dict= ast.literal_eval(llama_eval)
        print(llama_eval_dict)

        mark_dict[instance] = llama_eval_dict

    final_mark_dict[f"rule{i+1}"] = mark_dict

    i += 1

for rule_id, eval_dict in final_mark_dict.items():
    print(f"Average for {rule_id}:")

    mark_list = []
    for image_id, marks in eval_dict.items():
        mark_list.append(marks['total'])
    print(sum(mark_list)/len(mark_list))
    print(f"Number of instance: {len(mark_list)}")

GPT_correctly_predicted/rule1_correctly_predicted.json
0000005
The person on foot at the construction site is not wearing a hard hat, and the clothes do not cover the shoulders and legs completely. Shoes are not visible in the image.
The person on the left is not wearing a hard hat.
{'relevance': 2, 'equivalence': 2, 'specificity': 2, 'total': 6}
0000007
One worker is not wearing a hard hat, and another's clothes do not cover shoulders.
Multiple workers not wearing hard hats nor high-visibility vests working at night.
{'relevance': 2, 'equivalence': 1, 'specificity': 0, 'total': 3}
0000019
The worker on the left is not wearing a hard hat, and his clothes do not cover his shoulders.
Worker with a black cap and white shirt on the left is not wearing a hard hat.
{'relevance': 2, 'equivalence': 2, 'specificity': 2, 'total': 6}
0000039
The person on the left side of the image is not wearing a high-visibility vest, which is required for enhanced visibility and safety on construction sites.
T

## Bounding box

In [2]:
IoU = vqa_evaluator.bounding_box_score(prediction_file, label_file)
print(IoU)

0000001
{'0': 'No violations'}
0000002
{'1': {'reason': 'There is a person without a hard hat and the clothes do not cover shoulders and legs completely, and no high-visibility retroreflective vests are visible.', 'bounding_box': [0.25, 0.5, 0.35, 0.6]}, '4': {'reason': 'There are multiple workers in close proximity to the operating excavator, potentially in the blind spot of the operator.', 'bounding_box': [0.5, 0.6, 0.8, 0.9]}}
0000005
{'1': {'reason': 'The person on foot at the construction site is not wearing a hard hat, and the clothes do not cover the shoulders and legs completely. Shoes are not visible in the image.', 'bounding_box': [0.15, 0.55, 0.25, 0.95]}, '4': {'reason': 'The worker is walking in the potential blind spot of the excavator operator and within the operation radius of the excavator bucket.', 'bounding_box': [0.15, 0.55, 0.25, 0.95]}}
0000007
{'1': {'reason': "One worker is not wearing a hard hat, and another's clothes do not cover shoulders.", 'bounding_box': [